# CA3 — Demo

This notebook trains a simple REINFORCE policy on CartPole-v1 using the `src/` package. Cells are meant to be executed in order.

In [ ]:
# Imports and setup
import gym
import torch
import matplotlib.pyplot as plt
from src.config import Config
from src.model import MLPPolicy
from src.utils import set_seed, discounted_returns, returns_to_tensor, ensure_dir
from src.data import collect_episode
from src.losses import reinforce_loss, entropy_loss_from_logits

cfg = Config()
set_seed(cfg.seed)
env = gym.make(cfg.env_name)
obs_dim = env.observation_space.shape[0]
action_dim = env.action_space.n
policy = MLPPolicy(obs_dim, action_dim, hidden_sizes=cfg.hidden_sizes)
optimizer = torch.optim.Adam(policy.parameters(), lr=cfg.lr)
ensure_dir(cfg.save_dir)


In [ ]:
# Training loop (REINFORCE, Monte-Carlo returns)
returns_history = []
for ep in range(200):
    ep_data = collect_episode(env, policy, device='cpu', max_steps=cfg.max_steps_per_episode)
    rewards = ep_data['rewards']
    log_probs = torch.as_tensor(ep_data['log_probs'], dtype=torch.float32)
    G = discounted_returns(rewards, cfg.gamma)
    G_tensor = returns_to_tensor(G)
    loss = reinforce_loss(log_probs, G_tensor)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(policy.parameters(), cfg.max_grad_norm)
    optimizer.step()
    returns_history.append(sum(rewards))
    if (ep + 1) % 20 == 0:
        print(f'Episode {ep+1}	Return: {returns_history[-1]:.2f}')


In [ ]:
# Plotting returns (publication-quality figure)
plt.figure(figsize=(6,4))
plt.plot(returns_history, label='Episode Return')
plt.xlabel('Episode')
plt.ylabel('Return')
plt.title('REINFORCE on CartPole-v1')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig('../pictures/fig_01_convergence.png', dpi=300)
plt.show()


Notes:
- Adjust hyperparameters in `src/config.py`.
- To run on GPU, set `cfg.device` appropriately and move model/tensors to that device.
